# 奖励函数与数据准备

本节详细介绍 Wordle 的奖励函数设计和训练数据准备流程。

---

## 1. 奖励函数

Wordle 的奖励由 `wordle_reward.py` 中的 `compute_score` 函数计算，包含四个组件：

| 组件 | 分值范围 | 说明 |
|------|---------|------|
| `correct_answer` | 0 或 1.0 | 猜中秘密单词 |
| `partial_answer` | 0 - 0.8 | 部分匹配（0.2 * 绿色 + 0.1 * 黄色）|
| `length_bonus` | 0 - 1.0 | 猜中时轮数越少奖励越高（1 / 实际轮数）|
| `format_reward` | 0 - 0.2 | 使用 `<guess>[word]</guess>` 的回合占比（权重 0.2）|

奖励结构参考 Prime Intellect Verifiers 的 Wordle 环境，但保留了适配 verl 的差异：本 recipe 的 `partial_answer` 取整条轨迹中匹配度最高的一次猜词；`length_bonus` 和 `format_reward` 使用 Agent Loop 实际消耗的轮数，格式错误的回复也会占用一轮。四项权重与已验证实验保持不变。

### 奖励计算示例

```text
秘密单词: crane

Rollout A: 2 轮猜中
  correct_answer = 1.0
  partial_answer = 0.0  (猜中后不算 partial)
  length_bonus   = 1/2 = 0.5
  format_reward  = 1.0 * 0.2 = 0.2
  总奖励 = 1.0 + 0.0 + 0.5 + 0.2 = 1.7

Rollout B: 6 轮未猜中，历史最佳一次为 3 绿 1 黄
  correct_answer = 0.0
  partial_answer = 0.2*3 + 0.1*1 = 0.7
  length_bonus   = 0.0  (未猜中)
  format_reward  = 1.0 * 0.2 = 0.2
  总奖励 = 0.0 + 0.7 + 0.0 + 0.2 = 0.9
```

### 奖励函数返回值

`compute_score` 返回一个字典，包含总分和各组件分数：

```python
{
    'score': 1.7,         # 总分（用于训练）
    'correct': 1.0,       # 猜中奖励
    'partial': 0.0,       # 部分匹配
    'length_bonus': 0.5,  # 步数奖励
    'format': 1.0,        # 格式正确率
    'num_guesses': 2      # 猜测次数
}
```


验证时会分别记录每个组件到 tensorboard，方便分析模型的薄弱环节。

---

## 2. 数据准备

训练数据由 `prepare_data.py` 生成，词表来源于 TextArena Wordle-v0。

### 数据格式

每条数据包含以下字段：

```python
{
    'prompt': [
        {'role': 'system', 'content': 'You are a competitive game player...'},
        {'role': 'user', 'content': '[GAME] You are Playing Wordle...'}
    ],
    'raw_prompt': [...],      # 同 prompt
    'answer': 'crane',         # 秘密单词
    'index': 'wordle_train_0000_crane',
    'data_source': 'wordle',
    'reward_model': {'ground_truth': 'crane'}
}
```


### 生成命令

In [ ]:
%%bash
cd llm_rl/qwen3_wordle
python3 prepare_data.py --num_train 2000 --num_test 20 --output_dir data
python3 - <<'PY'
import pandas as pd
for path in ('data/wordle_train.parquet', 'data/wordle_test.parquet'):
    frame = pd.read_parquet(path)
    print(path, frame.shape, 'unique_index=', frame['index'].is_unique)
PY


1. （判断题）Wordle 奖励函数中，猜中后不再计算 partial_answer 分数。

2. （判断题）length_bonus 的值与猜测次数成反比，猜得越快奖励越高。

3. （判断题）format_reward 的权重是 1.0，与 correct_answer 相同。

4. （单选题）一个 2 轮猜中的 rollout，其 length_bonus 是多少？
    A. 0.2
    B. 0.5
    C. 1.0
    D. 2.0

5. （单选题）训练数据的词表来源于哪里？
    A. Hugging Face 数据集
    B. TextArena Wordle-v0
    C. 手动标注
    D. Wordle 官方网站

6. （多选题）Wordle 奖励函数包含以下哪些组件？
    A. correct_answer
    B. partial_answer
    C. length_bonus
    D. format_reward

In [ ]:
!cat ../cann-learning-hub/tutorials/rl_training_pipeline/03_wordle_rl_training/answer/03.03_answer.txt